In [19]:
import json
import os

from dotenv import load_dotenv
import httpx
import openai
from openai.types.chat import ChatCompletionMessage

load_dotenv()

MODEL = "gpt-4o-mini"
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

SYSTEM_PROMPT = """
You are a Movie Expert agent backed by the Nomad Movies API.

Tools:
- get_popular_movies — list popular movies (GET /movies). Use when the user asks what is popular or trending now.
- get_movie_details — details for one movie by numeric id (GET /movies/:id). Use when the user asks about a specific movie by ID.
- get_movie_credits — cast and crew (GET /movies/:id/credits). Use when the user asks who stars, who directed, or credits for a movie ID.

Call the appropriate tool to fetch data. After you receive tool results, answer the user in natural language using that data. Match the user’s language (e.g. Korean if they wrote in Korean).
"""

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Fetch popular movies from the /movies endpoint.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Fetch movie information for a given movie id.",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "Movie id."}},
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Fetch cast and crew for a given movie id.",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "Movie id."}},
                "required": ["id"],
            },
        },
    },
]


def get_popular_movies() -> object:
    r = httpx.get(f"{BASE_URL}/movies", timeout=5.0)
    r.raise_for_status()
    return r.json()


def get_movie_details(movie_id: int) -> object:
    r = httpx.get(f"{BASE_URL}/movies/{movie_id}", timeout=5.0)
    r.raise_for_status()
    return r.json()


def get_movie_credits(movie_id: int) -> object:
    r = httpx.get(f"{BASE_URL}/movies/{movie_id}/credits", timeout=5.0)
    r.raise_for_status()
    return r.json()


def execute_tool(name: str, arguments_json: str) -> str:
    try:
        args = json.loads(arguments_json)
        if name == "get_popular_movies":
            data = get_popular_movies()
        elif name == "get_movie_details":
            data = get_movie_details(int(args["id"]))
        elif name == "get_movie_credits":
            data = get_movie_credits(int(args["id"]))
        else:
            return json.dumps({"error": f"unknown tool: {name}"})
        return json.dumps(data)
    except Exception as e:
        return json.dumps({"error": type(e).__name__, "message": str(e)})


def _assistant_message_dict(msg: ChatCompletionMessage) -> dict:
    d: dict = {"role": "assistant"}
    if msg.content:
        d["content"] = msg.content
    if msg.tool_calls:
        d["tool_calls"] = [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }
            for tc in msg.tool_calls
        ]
    return d


def print_conversation(messages: list) -> None:
    print()
    print("=" * 60)
    for m in messages:
        if m["role"] == "system":
            continue
        if m["role"] == "user":
            print(f"You: {m['content']}")
        elif m["role"] == "assistant":
            if m.get("content"):
                print(f"Assistant: {m['content']}")
            if m.get("tool_calls"):
                for tc in m["tool_calls"]:
                    fn = tc["function"]["name"]
                    arg = tc["function"]["arguments"]
                    print(f"- [tool] {fn} {arg}")


def run_agent(user_content: str, messages: list | None = None) -> str:
    client = openai.OpenAI()
    if messages is None:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.append({"role": "user", "content": user_content})
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",
    )
    msg = resp.choices[0].message
    if not msg.tool_calls:
        text = msg.content or ""
        messages.append({"role": "assistant", "content": text})
        print(f"Assistant: {text}")
        return text
    messages.append(_assistant_message_dict(msg))
    for tc in msg.tool_calls:
        out = execute_tool(tc.function.name, tc.function.arguments)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})
        print(f"- [tool] {tc.function.name} {tc.function.arguments}")
    resp2 = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="none",
    )
    text2 = resp2.choices[0].message.content or ""
    messages.append({"role": "assistant", "content": text2})
    print(f"Assistant: {text2}")
    return text2

In [20]:
# Run the previous cell first. History is kept in `messages`.
# End with: quit / exit / q

messages = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Movie Expert — type 'quit' to exit.\n")

while True:
    print("=" * 60)
    user_message = input("You: ").strip()
    print(f"You: {user_message}")
    if user_message.lower() in ("quit", "exit", "q"):
        print("Bye.")
        break
    if not user_message:
        continue
    run_agent(user_message, messages=messages)

Movie Expert — type 'quit' to exit.

You: 지금 인기 있는 영화가 무엇인지 알려줘
- [tool] get_popular_movies {}
Assistant: 현재 인기 있는 영화들은 다음과 같습니다:

1. **Your Heart Will Be Broken**
   - 개봉일: 2026년 3월 26일
   - 줄거리: 고등학생 폴리나는 학교에서 괴롭힘을 당하지 않기 위해 주범인 바르스와 남자친구인 척하는 계약을 맺습니다. 이 과정에서 진정한 감정이 생기게 되지만, 그녀의 가족과 친구들은 이들을 갈라놓고자 합니다.
   - 평점: 6.6
   - ![포스터](https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg)

2. **Avatar: Fire and Ash**
   - 개봉일: 2025년 12월 17일
   - 줄거리: Jake Sully와 Neytiri는 아쉬족이라는 새로운 적과 맞서 싸워야 합니다. 가족과 판도라의 미래를 위해 치열한 전투를 벌입니다.
   - 평점: 7.4
   - ![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)

3. **Hoppers**
   - 개봉일: 2026년 3월 4일
   - 줄거리: 과학자들이 사람의 의식을 로봇 동물로 '홉'할 수 있는 발견을 하면서, 한 동물 사랑하는 여인이 동물과의 신비로운 소통을 시도합니다.
   - 평점: 7.6
   - ![포스터](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)

4. **Crime 101**
   - 개봉일: 2026년 2월 11일
   - 줄거리: 로스앤젤레스에서 발생하는 고위험 범죄들과 얽히게 되는 도둑과 보험 중개인의 이야기를 다룹니다.
   - 평점: 7.1
   - ![포스터](https://image.tmdb.org/t/p/w78